# Nghiên Cứu Thử Nghiệm Đa Mô Hình (Multi-Model Pilot Benchmark - IEEE TGRS)

Notebook này đánh giá lần lượt **TOÀN BỘ 5 mô hình thị giác - ngôn ngữ** được quy định trong Kế hoạch nghiên cứu IEEE TGRS trên tập mẫu con chuẩn hóa **1.000 câu hỏi phân tầng (stratified)**:
1. **Qwen2.5-VL-7B-Instruct** ($P=32$ px/token, SigLIP2)
2. **InternVL2.5-8B** ($P=28$ px/token, InternViT)
3. **LLaVA-1.5-7B** ($P=14$ px/token, CLIP ViT@336)
4. **LLaVA-OneVision-7B** ($P=14$ px/token / ô, SigLIP@384 AnyRes)
5. **GeoChat-7B** ($P=14$ px/token, CLIP ViT@504 - Remote Sensing Specialized)

Sau khi chạy xong, hệ thống tự động tổng hợp **BẢNG SO SÁNH ĐA MÔ HÌNH** kiểm định các giả thuyết:
- **H1**: Nút thắt token độc lập nút thắt pixel (Hệ số $\beta_1 < 0$)
- **H2**: Hai ngưỡng phân giải độc lập ($p_0^{50}$ cảm biến và $\rho_{tok}^*$ token)
- **H3**: Bậc thang Johnson (Kiểm định Jonckheere-Terpstra monotonic trend)
- **H4**: Hiệu năng dự đoán có chọn lọc RGSP vs các baselines (Cov@10%, AURC, Abst.-acc trên U)

In [ ]:
# [1] Cài đặt các thư viện MLLM & công cụ khoa học
!pip install -q "transformers>=4.49.0" "accelerate>=0.26.0" "qwen-vl-utils>=0.0.8" torchvision scipy pillow timm einops

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")

In [ ]:
# [2] Cập nhật mã nguồn mới nhất từ GitHub
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/rgsp_nckh")
if not WORK_DIR.exists():
    !git clone https://github.com/kttt294/RGSP-nckh.git {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull origin main

sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / "rs-solve"))
sys.path.insert(0, str(WORK_DIR / "rs-solve" / "eval"))
print("Đã nạp môi trường mã nguồn thành công!")

In [ ]:
# [3] Cấu hình thực nghiệm Đa Mô Hình Pilot Test
BENCHMARK_FILE = WORK_DIR / "kaggle_benchmark_output" / "rs_solve_full_benchmark_en.jsonl"
IMAGE_ROOT = Path("/kaggle/input")
OUTPUT_PREDS_DIR = Path("/kaggle/working/eval_outputs")
OUTPUT_REPORTS_DIR = Path("/kaggle/working/eval_reports")

OUTPUT_PREDS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# 1.000 mẫu phân tầng đều qua các loại câu hỏi (Q1 - Q6)
PILOT_SAMPLES = 1000

# Danh sách 5 mô hình trong bài báo
MODELS_CONFIG = [
    {"name": "qwen2.5-vl", "p_patch": 32, "label": "Qwen2.5-VL-7B (P=32)"},
    {"name": "internvl", "p_patch": 28, "label": "InternVL2.5-8B (P=28)"},
    {"name": "llava-1.5", "p_patch": 14, "label": "LLaVA-1.5-7B (P=14)"},
    {"name": "llava-onevision", "p_patch": 14, "label": "LLaVA-OneVision-7B (P=14)"},
    {"name": "geochat", "p_patch": 14, "label": "GeoChat-7B (P=14)"}
]

print(f"Số lượng mẫu pilot : {PILOT_SAMPLES:,} câu (Stratified)")
print(f"Số lượng mô hình   : {len(MODELS_CONFIG)} mô hình")

In [ ]:
# [4] Vòng lặp suy luận tuần tự từng mô hình & giải phóng VRAM an toàn
import gc
from run_inference import get_model_adapter, run_evaluation

generated_pred_files = []
device = "cuda" if torch.cuda.is_available() else "cpu"

for idx, cfg in enumerate(MODELS_CONFIG, 1):
    m_name = cfg["name"]
    m_label = cfg["label"]
    clean_name = m_name.replace("-", "_").replace(".", "_")
    pred_out = OUTPUT_PREDS_DIR / f"preds_{clean_name}.jsonl"
    
    print("\n" + "=" * 80)
    print(f"[{idx}/{len(MODELS_CONFIG)}] KHỞI CHẠY ĐÁNH GIÁ MÔ HÌNH: {m_label}")
    print("=" * 80)
    
    try:
        adapter = get_model_adapter(m_name, device=device)
        run_evaluation(
            model_adapter=adapter,
            benchmark_file=BENCHMARK_FILE,
            image_root=IMAGE_ROOT,
            output_file=pred_out,
            max_samples=PILOT_SAMPLES,
            resume=True,
            stratified=True
        )
        generated_pred_files.append(pred_out)
        # Thu hồi hoàn toàn bộ nhớ GPU để tránh OOM mô hình kế tiếp
        adapter.unload()
    except Exception as e:
        print(f"[!] CẢNH BÁO: Lỗi khi chạy {m_name}: {e}")
        import traceback
        traceback.print_exc()
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"Bộ nhớ VRAM còn trống: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")

print("\n" + "=" * 80)
print(f"HOÀN TẤT SUY LUẬN {len(generated_pred_files)} / {len(MODELS_CONFIG)} MÔ HÌNH!")
print("=" * 80)

In [ ]:
# [5] Tính toán toàn bộ Bảng chỉ số IEEE TGRS và BẢNG TỔNG HỢP SO SÁNH ĐA MÔ HÌNH
from compute_metrics import evaluate_prediction_file, generate_cross_model_comparison_report
from IPython.display import Markdown, display

all_model_results = []
for p_file in generated_pred_files:
    fname = p_file.name.lower()
    if "intern" in fname:
        p = 28
    elif "llava" in fname or "geochat" in fname:
        p = 14
    else:
        p = 32
    
    res = evaluate_prediction_file(p_file, OUTPUT_REPORTS_DIR, p_patch=p)
    if res:
        all_model_results.append(res)

# Xuất Master Comparison Table nếu có từ 2 mô hình trở lên
if len(all_model_results) >= 2:
    cmp_file = generate_cross_model_comparison_report(all_model_results, OUTPUT_REPORTS_DIR)
    if cmp_file.exists():
        display(Markdown(cmp_file.read_text(encoding="utf-8")))

In [ ]:
# [6] Nén toàn bộ kết quả thí nghiệm để tải về lưu trữ
!tar -czvf /kaggle/working/eval_multimodel_pilot_results.tar.gz /kaggle/working/eval_outputs /kaggle/working/eval_reports
print("Đã lưu gói báo cáo tại /kaggle/working/eval_multimodel_pilot_results.tar.gz")